# Финальная модель: PP-OCRv6 small

Единственный рабочий распознаватель. Реальный прогон по 1984 размеченным изображениям
и 9 целевым вырезкам, из которых 7 имеют читаемую визуальную метку. Без обучения,
подгонки ответов и платных API. Выбор модели обоснован в [отчёте](../README.md).
Другие модели исследованы в истории Git и не входят в текущий код.

In [1]:
from pathlib import Path
import sys,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'captcha_lab').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import pandas as pd
from captcha_lab.evaluation import manifest,run_benchmark,summary_frame,RESULTS


## Готовая размеченная выборка

Изображения лежат в репозитории обычными JPG/PNG, архивов и распаковки нет.
Сохранены ведущие нули. Метки публичного набора предоставлены датасетом; target
визуально размечен ассистентом и не является независимым production-test.

In [2]:
labels=manifest('holdout',require_images=True)
assert len(labels)==1984
print('Holdout:',len(labels),'Target:',len(manifest('target',require_images=True)))
display(labels[['image_id','label','image_path']].head(12))
display(pd.DataFrame([json.loads((ROOT/'research/model.json').read_text(encoding='utf-8'))]))

Holdout: 1984 Target: 9


,image_id,label,image_path
0,test_000c1b6d4e5f11fb75d55a7c094b901e,161660,datasets/holdout/images/161660.000c1b6d4e5f11f...
1,test_00105a4c053053509955d95368312987,496670,datasets/holdout/images/496670.00105a4c0530535...
2,test_001a3b40ef793b5984df8969f145d7bb,744782,datasets/holdout/images/744782.001a3b40ef793b5...
3,test_0021e642491cf5bbc13cfc59fa46084a,017232,datasets/holdout/images/017232.0021e642491cf5b...
4,test_0047646a0e3b67ebc360e43ba88d1596,853091,datasets/holdout/images/853091.0047646a0e3b67e...
5,test_0064b2188592a16991d5394b3ec5bf01,959686,datasets/holdout/images/959686.0064b2188592a16...
6,test_00a5d81ab5060a0d45608169ce20bda9,159645,datasets/holdout/images/159645.00a5d81ab5060a0...
7,test_00e02c7c9da2cc0d3cdf5e3d9183a9f5,952376,datasets/holdout/images/952376.00e02c7c9da2cc0...
8,test_0123f4327a176c460aeefecd1d69bc11,374639,datasets/holdout/images/374639.0123f4327a176c4...
9,test_0127972a796c1f306fdb5f919a7649fa,517493,datasets/holdout/images/517493.0127972a796c1f3...


,name,backend,filename,sha256,bytes,detector,classifier,ctc,cpu_threads,model_card
0,PP-OCRv6 small recognition,RapidOCR 3.10.0 / ONNX Runtime 1.31.0,PP-OCRv6_rec_small.onnx,6f327246b50388f3c176ae304bd95767ea6dc0c9ae9215...,21234383,False,False,True,2,https://huggingface.co/PaddlePaddle/PP-OCRv6_s...


## Настоящий локальный inference

Ячейка заново загружает единственную модель и распознаёт все картинки. Время включает
открытие файла, preprocessing, inference и decoding; инициализация и прогрев отдельно.
Ответы и эталоны сохраняются вместе, в одном CSV на набор. Прогрев — белое изображение.
Вход модели содержит только пиксели, без имени файла и эталона.

In [3]:
frames=run_benchmark()

ppocr-small holdout 200 / 1984


ppocr-small holdout 400 / 1984


ppocr-small holdout 600 / 1984


ppocr-small holdout 800 / 1984


ppocr-small holdout 1000 / 1984


ppocr-small holdout 1200 / 1984


ppocr-small holdout 1400 / 1984


ppocr-small holdout 1600 / 1984


ppocr-small holdout 1800 / 1984


In [4]:
summary=summary_frame()
summary.to_csv(RESULTS/'summary.csv',index=False)
display(summary)
for dataset,frame in frames.items():
    print(dataset,': примеры ошибок')
    display(frame[frame.correct.eq(False)][['image_id','label','prediction','error']].head(15))

,model,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
0,ppocr-small,holdout,1984,1876,0.945565,0.934693,0.954714,0.010165,0,0,25.4297,33.272065,25.4297,33.272065
1,ppocr-small,target,7,4,0.571429,0.250458,0.841780,0.190476,0,0,27.8722,30.670950,27.8722,30.670950


holdout : примеры ошибок


,image_id,label,prediction,error
40,test_059a19ca75774a6ff857551a862a9eb9,772537,712537,
55,test_0793dc242a40b91872d09106dea3cb64,173367,73367,
72,test_0a5a2dad9d0c93d13c721af538dc74f3,677799,77799,
84,test_0babecfa8fc8a18b3726c0ef3d5197d1,844073,844073-,
85,test_0bb791f343df3cd5fe35ec5f6be942df,071887,71887,
131,test_10aa0ed8771dcd750c1b1e26f00e6af2,538261,538261.,
143,test_112da6d0e0769450fbc3248d7681f1ea,594026,594026.,
150,test_11a26c470c305ef4f9922e36ef7c1635,579877,579877.,
151,test_11e687de5eca9c8e1fee48660bd2c497,020684,20684,
153,test_121e092360cac568f640e4e048d1470b,592494,392494,


target : примеры ошибок


,image_id,label,prediction,error
1,33ef91d464c03eb9,894412,8048,
2,41ada5001f1e1e88,432077,432071,
7,d4c0652824fb2dfd,639125,60926,


## Вывод

Основная метрика — полное совпадение строки. Одна неправильная цифра означает ошибку.
Wilson-интервал отражает размер выборки; ошибки распознавания входят в знаменатель,
два спорных target-примера исключаются. Высокая точность публичного набора не доказывает
перенос на предоставленное семейство капч. Полный разбор и ограничения — в отчёте.